In [ ]:
from datasets import load_dataset

In [14]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel

In [16]:
from tokenizers.pre_tokenizers import Whitespace

In [18]:
from tokenizers.trainers import WordLevelTrainer

In [51]:
from pathlib import Path

In [7]:
train_dataset = load_dataset(
    path="Helsinki-NLP/opus-100",
    name="en-fa",
    split="train",
)

val_dataset = load_dataset(
    path="Helsinki-NLP/opus-100",
    name="en-fa",
    split="validation",
)

In [58]:
len(train_dataset)

1000000

In [11]:
print(train_dataset[0])

{'translation': {'en': 'Pack your stuff.', 'fa': 'وسايلتو جمع کن باشه.'}}


In [12]:
print(val_dataset[0])

{'translation': {'en': 'And they say: There is naught but our life of the world; we die and we live, and naught destroyeth us save time; when they have no knowledge whatsoever of (all) that; they do but guess.', 'fa': 'و گفتند: جز زندگى دنيوى ما هيچ نيست. مى\u200cميريم و زنده مى\u200cشويم و ما را جز دهر هلاك نكند. آنان را بدان دانشى نيست و جز در پندارى نيستند.'}}


In [39]:
def get_single_sentence(dataset, language):
    for data in dataset:
        yield data["translation"][language]

In [45]:
def sentence_generator(dataset, language: str):
    for data in dataset:
        yield data["translation"][language]

def create_or_load_tokenizer(dataset, language: str, file_path:str=None):
    if file_path is not None:
        if Path(file_path).exists():
            tokenizer = Tokenizer.from_file(file_path)
            return tokenizer
    
    trainer = WordLevelTrainer(
        min_frequency=2,
        show_progress=True,
        special_tokens=["<SOS>", "<EOS>", "<PAD>", "<unk>"],
    )
    tokenizer = Tokenizer(WordLevel(unk_token="<unk>"))
    tokenizer.pre_tokenizer = Whitespace()
    tokenizer.train_from_iterator(sentence_generator(dataset, language), trainer=trainer)
    tokenizer.save(file_path)
    return tokenizer

In [46]:
english_tokenizer = create_or_load_tokenizer(dataset, "en")
farsi_tokenizer = create_or_load_tokenizer(dataset, "fa")

In [57]:
english_tokenizer.encode("Hello how are you").ids

[503, 135, 33, 8]

In [65]:
print(english_tokenizer.token_to_id("<SOS>"))
print(english_tokenizer.token_to_id("<EOS>"))
print(english_tokenizer.token_to_id("<unk>"))
print(english_tokenizer.token_to_id("<PAD>"))

0
1
3
2


In [48]:
farsi_tokenizer.encode("سلام حال شما خوب است").ids

[146, 136, 28, 81, 18]

In [49]:
farsi_tokenizer.encode("سلام حال شما چطور است").ids

[146, 136, 28, 252, 18]

In [ ]:
def create_data(
    dataset_path:str,
    dataset_name:str,
    split:str,
    language: str,
    file_path:str,
):
    dataset = load_dataset(
        path=dataset_path,
        name=dataset_name,
        split="train",
    )
    tokenizer = create_or_load_tokenizer(dataset, language, file_path)
    vocab_size = tokenizer.get_vocab_size()

    max_seq_len = 0
    for data in dataset:
        token_ids_ls = tokenizer.encode(data).ids
        max_seq_len = max(max_seq_len, len(token_ids_ls))

    seq_len = max_seq_len + 2
        
    
    # Create a BilingualDataset (your torch Dataset) wrapping the raw data and tokenizers
    
    # Pass the BilingualDataset into a DataLoader
    